# Phase 4 — 8 min / 32 calls (middle ground)

Probe 28 calls solved 2/13; calls40 solved 1/13 and projected 12.7 h. Test 32 calls to see if a small increase helps without over-editing.

## 1. Environment

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Load primary and create 32-call variant

In [ ]:
from pathlib import Path
import re, shutil, zipfile

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

SRC_PRIMARY = Path('/kaggle/input/datasets/utsavgupta1111/gemma4-primary')
if not SRC_PRIMARY.exists():
    hits = list(Path('/kaggle/input').rglob('gemma4-primary'))
    assert hits, 'gemma4-primary dataset not attached'
    SRC_PRIMARY = hits[0]

PRIMARY_DIR = WORKING_DIR / 'submission'
CALLS32_DIR = WORKING_DIR / 'submission-calls32'

def extract_agent(src: Path, dst: Path) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True)
    zips = list(src.rglob('*.zip'))
    if zips:
        print('extracting', zips[0])
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(dst)
        return
    agents = list(src.rglob('agent.yaml'))
    assert agents, f'agent.yaml not found under {src}'
    root = agents[0].parent
    print('copying from', root)
    shutil.copytree(root, dst, dirs_exist_ok=True)

extract_agent(SRC_PRIMARY, PRIMARY_DIR)
assert (PRIMARY_DIR / 'agent.yaml').is_file()

# 8 min / 32 calls / 4096 thinking
if CALLS32_DIR.exists():
    shutil.rmtree(CALLS32_DIR)
shutil.copytree(PRIMARY_DIR, CALLS32_DIR)
(CALLS32_DIR / 'eval_config.yaml').write_text(
    "evaluation:\n"
    "  timeout_seconds: 240\n"
    "  max_tool_calls: 32\n"
    "  max_time_minutes: 8\n"
    "  max_turns: 80\n"
)
(CALLS32_DIR / 'configs/sampling.yaml').write_text(
    "temperature: 0.2\n"
    "top_p: 0.95\n"
    "max_output_tokens: 8192\n"
    "thinking_config:\n"
    "  thinking_budget: 4096\n"
    "  include_thoughts: true\n"
)
p = CALLS32_DIR / 'prompts/system.md'
p.write_text(p.read_text().replace('24 tool calls and 5 minutes', '32 tool calls and 8 minutes'))

from swegemma.models import load_tasks
tasks = load_tasks(TASKS_PATH)
by_repo = {}
for t in tasks:
    by_repo.setdefault(t.repo, []).append(t)
SELECTED_TASKS = [t for ts in by_repo.values() for t in ts[:4]]

AGENT_DIR = PRIMARY_DIR  # vLLM model/adapters
print('calls32:', sorted(x.name for x in CALLS32_DIR.iterdir()))
print('selected:', len(SELECTED_TASKS), [t.instance_id for t in SELECTED_TASKS])


## 3. Start vLLM

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 4. Run 8 min / 32 calls

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


def evaluate_bundle(tag: str, agent_dir: Path) -> pd.DataFrame:
    eval_config_file = next(
        (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := agent_dir / name).is_file()),
        None,
    )
    raw = yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {} if eval_config_file else {}
    sec = raw.get('evaluation', raw)
    timeout_seconds = int(sec.get('timeout_seconds', 300))
    verification_timeout_seconds = int(sec.get('verification_timeout_seconds', 300))
    max_tool_calls = int(sec.get('max_tool_calls', 100))
    max_time_minutes = float(sec.get('max_time_minutes', 60.0))
    turns_raw = sec.get('max_turns', sec.get('max_llm_calls'))
    max_turns = int(turns_raw) if turns_raw is not None else None

    limits, gen_constraints = build_submission_limits()
    cfg = EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=WORKING_DIR / f'results-{tag}',
        submission_dir=agent_dir,
        models=models,
        sandbox='subprocess',
        timeout_seconds=timeout_seconds,
        verification_timeout_seconds=verification_timeout_seconds,
        max_time_minutes=max_time_minutes,
        max_tool_calls=max_tool_calls,
        max_turns=max_turns,
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15,
            overlap_size=2,
            token_threshold=14336,
            event_retention_size=5,
        ),
        graph_dir=GRAPH_DIR,
        embeddings_dir=EMBEDDINGS_DIR,
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    )
    evaluator = Evaluator(cfg)
    rows = []
    for idx, task in enumerate(SELECTED_TASKS, start=1):
        print(f'[{tag} {idx}/{len(SELECTED_TASKS)}] {task.instance_id} ({task.repo})...')
        result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SELECTED_TASKS))
        patch = (result.agent_patch or '').strip()
        rows.append({
            'id': task.instance_id,
            'repo': task.repo,
            'prediction': patch if patch else 'NO_PATCH',
            'resolved': bool(result.resolved),
            'exit_code': result.test_exit_code,
            'patch_chars': len(patch),
            'tool_calls': result.tool_calls,
            'duration_s': result.duration_seconds,
        })
        print(f'  -> resolved={result.resolved}, patch_chars={len(patch)}, tool_calls={result.tool_calls}, duration={result.duration_seconds:.1f}s')

    df = pd.DataFrame(rows)
    df.to_csv(WORKING_DIR / f'task-results-{tag}.csv', index=False)
    df[['id', 'prediction']].to_csv(WORKING_DIR / f'predictions-{tag}.csv', index=False)
    return df


calls32 = evaluate_bundle('calls32', CALLS32_DIR)


## 5. Compare

In [ ]:
import pandas as pd
from pathlib import Path

frames = {'calls32': calls32}
for tag in ['primary', 'probe', 'calls40', 'v2']:
    p = Path(f'/kaggle/working/task-results-{tag}.csv')
    if p.exists():
        frames[tag] = pd.read_csv(p)

for name, df in frames.items():
    print(f'\n=== {name} ===')
    print('resolved:', int(df.resolved.sum()), '/', len(df))
    print(df.groupby('repo').resolved.agg(['sum', 'count']))
    print('non-empty patches:', int((df.prediction != 'NO_PATCH').sum()), '/', len(df))
    print('duration p50/p95:', round(df.duration_s.quantile(.5), 1), round(df.duration_s.quantile(.95), 1))
    print('mean tool calls:', round(df.tool_calls.mean(), 1), 'max:', int(df.tool_calls.max()))
    print('NO_PATCH:', df.loc[df.prediction == 'NO_PATCH', 'id'].tolist())


## 6. Stop server

In [ ]:
server_instance.stop()
print("vLLM stopped")